# **Load the Dataset**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install --upgrade objaverse
!pip install torch torchvision torchaudio trimesh numpy objaverse tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.7 MB/s eta 0:00:00
  Created wheel for gputil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=37d826485ad90dd458472b67dadbc31aba08e5297ff6754bc03465305b36e25c
  Stored in directory: /root/.cache/pip/wheels/2b/4d/8f/55fb4f7b9b591891e8d3f72977c4ec6c7763b39c19f0861595
Successfully built gputil
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# **Load the Components**

In [3]:
%matplotlib inline
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import objaverse
import trimesh
import gc
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Define device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {DEVICE}")

In [4]:
uuid_embeddings = np.load("/content/drive/MyDrive/Project/df_english.npy", allow_pickle=True)

In [5]:
uuid_embeddings

array([['bfd459469a9449978ff5fc7199d5848b',
        "3D Mandala #2 - The Power of Love went a bit weird with this one.. in the centre a blue female is punching the red male, breaking his heart in the process, triggering feelings like he's being anally probed by aliens, sending shocks through his eyeball, releasing some tears."],
       ['a4bb737d810049c98680a53262552a7b',
        "Glass 眼睛 A Glass which don't have shader and i don't know why"],
       ['01474f78a328473b8d99ded65effb2cc',
        'Low Poly Barrel super simple barrel'],
       ...,
       ['69925e5dab364b4ba87b036f6042c442',
        'Consolevert4 SciFi Controls \n\nFree to use in commercial and personal projects!\n\nEnjoy!'],
       ['63b0f1e3ccdd4b3fabec4443a2b25cd8',
        'Neo-Normcore Collection, Insulin Placebo Human Insulin Hexamer by model3dbiology is licensed under CC Attribution'],
       ['168bb415db3247a19e5fcc4d23c26c91',
        "R.O.Y. Bot Zbrush, Go-Z'd to C4D create OVDB Bevels and baked in 3D Coat. Sci

In [6]:
uuid_embeddings.shape

(236277, 2)

In [7]:
uuid_embeddings[0, 1]

"3D Mandala #2 - The Power of Love went a bit weird with this one.. in the centre a blue female is punching the red male, breaking his heart in the process, triggering feelings like he's being anally probed by aliens, sending shocks through his eyeball, releasing some tears."

In [8]:
import numpy as np
import objaverse
import os
import trimesh
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def generate_triplane_from_mesh(obj_path, triplane_resolution=128, voxel_resolution=128, device='cuda'):
    try:
        mesh = trimesh.load(obj_path, force='mesh', process=False)
        if isinstance(mesh, trimesh.Scene):
            mesh = mesh.dump(concatenate=True)
        if not isinstance(mesh, trimesh.Trimesh):
            logger.error(f"Invalid mesh at {obj_path}: not a Trimesh")
            return None
        if len(mesh.vertices) == 0 or len(mesh.faces) == 0:
            logger.error(f"Empty mesh at {obj_path}")
            return None

        center = mesh.bounds.mean(axis=0)
        mesh.apply_translation(-center)
        max_extent = np.ptp(mesh.bounds, axis=0).max()
        if max_extent > 1e-6:
            mesh.apply_scale(1.0 / max_extent)

        pitch = 1.0 / voxel_resolution
        voxel_grid = mesh.voxelized(pitch=pitch)
        voxel_matrix = voxel_grid.matrix.astype(np.float32)

        current_shape = voxel_matrix.shape
        target_shape = (voxel_resolution, voxel_resolution, voxel_resolution)
        padded_matrix = np.zeros(target_shape, dtype=np.float32)
        min_shape = tuple(min(s, t) for s, t in zip(current_shape, target_shape))
        padded_matrix[:min_shape[0], :min_shape[1], :min_shape[2]] = \
            voxel_matrix[:min_shape[0], :min_shape[1], :min_shape[2]]
        voxel_matrix = padded_matrix

        if voxel_matrix.sum() == 0:
            logger.error(f"Empty voxel grid for {obj_path}")
            return None

        plane_xy = np.max(voxel_matrix, axis=2)
        plane_yz = np.max(voxel_matrix, axis=0)
        plane_xz = np.max(voxel_matrix, axis=1)

        plane_xy_t = torch.from_numpy(plane_xy).unsqueeze(0).float().to(device)
        plane_yz_t = torch.from_numpy(plane_yz).unsqueeze(0).float().to(device)
        plane_xz_t = torch.from_numpy(plane_xz).unsqueeze(0).float().to(device)

        target_size = (triplane_resolution, triplane_resolution)
        plane_xy_t = F.interpolate(plane_xy_t.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)
        plane_yz_t = F.interpolate(plane_yz_t.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)
        plane_xz_t = F.interpolate(plane_xz_t.unsqueeze(0), size=target_size, mode='bilinear', align_corners=False).squeeze(0)

        plane_xy_t = torch.clamp(plane_xy_t, 0.0, 1.0)
        plane_yz_t = torch.clamp(plane_yz_t, 0.0, 1.0)
        plane_xz_t = torch.clamp(plane_xz_t, 0.0, 1.0)

        return plane_xy_t.cpu(), plane_yz_t.cpu(), plane_xz_t.cpu()
    except Exception as e:
        logger.error(f"Failed to process mesh at {obj_path}: {e}")
        return None

def generate_and_save_triplanes(caption_data_path, output_dir=None, start_index=0, end_index=None, triplane_resolution=128, voxel_resolution=128, device='cuda'):
    if output_dir is None:
        output_dir = os.path.dirname(caption_data_path)
    os.makedirs(output_dir, exist_ok=True)

    try:
        caption_data = np.load(caption_data_path, allow_pickle=True)
        logger.info(f"Loaded caption data with shape: {caption_data.shape}")
    except Exception as e:
        logger.error(f"Failed to load caption data: {e}")
        return

    max_index = caption_data.shape[0]
    if start_index < 0 or start_index >= max_index:
        logger.error(f"start_index {start_index} is out of bounds [0, {max_index})")
        return
    if end_index is None:
        end_index = max_index
    if end_index <= start_index or end_index > max_index:
        logger.error(f"end_index {end_index} is invalid; must be > {start_index} and <= {max_index}")
        return

    uuids = caption_data[start_index:end_index, 0]
    captions = caption_data[start_index:end_index, 1]

    for uuid, caption in tqdm(zip(uuids, captions), total=len(uuids), desc=f"Generating triplanes [{start_index}:{end_index}]"):
        try:
            objects = objaverse.load_objects(uids=[uuid])
            if uuid not in objects:
                logger.warning(f"Model {uuid} not found in Objaverse")
                continue
            obj_path = objects[uuid]
            if not isinstance(obj_path, str) or not os.path.exists(obj_path):
                logger.error(f"Invalid or missing file for {uuid}: {obj_path}")
                continue

            triplane = generate_triplane_from_mesh(obj_path, triplane_resolution, voxel_resolution, device)
            if triplane is None:
                logger.warning(f"Failed to generate triplane for {uuid}")
                continue

            plane_xy, plane_yz, plane_xz = triplane
            data = {
                'plane_xy': plane_xy.numpy(),
                'plane_yz': plane_yz.numpy(),
                'plane_xz': plane_xz.numpy(),
                'caption': caption
            }

            output_path = os.path.join(output_dir, f"{uuid}.npy")
            np.save(output_path, data)
            logger.info(f"Saved triplane and caption for {uuid} to {output_path}")

            del triplane, plane_xy, plane_yz, plane_xz, data
            torch.cuda.empty_cache() if device == 'cuda' else None

        except Exception as e:
            logger.error(f"Failed to process model {uuid}: {e}")
            continue

class TriplaneTextFromFilesDataset(Dataset):
    def __init__(self, triplane_dir, num_models=None, device='cuda'):
        self.triplane_dir = triplane_dir
        self.device = device

        self.npy_files = [f for f in os.listdir(triplane_dir) if f.endswith('.npy')]
        if not self.npy_files:
            logger.error(f"No .npy files found in {triplane_dir}")
            raise ValueError(f"No .npy files found in {triplane_dir}")

        if num_models is not None:
            self.npy_files = self.npy_files[:min(num_models, len(self.npy_files))]

        self.valid_files = []
        self.uuids = []

        for npy_file in tqdm(self.npy_files, desc="Validating .npy files"):
            try:
                file_path = os.path.join(self.triplane_dir, npy_file)
                data = np.load(file_path, allow_pickle=True).item()
                if not all(key in data for key in ['plane_xy', 'plane_yz', 'plane_xz', 'caption']):
                    logger.warning(f"Invalid .npy file {npy_file}: missing required keys")
                    continue
                if data['plane_xy'].shape != (1, 128, 128) or \
                   data['plane_yz'].shape != (1, 128, 128) or \
                   data['plane_xz'].shape != (1, 128, 128):
                    logger.warning(f"Invalid .npy file {npy_file}: incorrect triplane shapes")
                    continue
                self.valid_files.append(npy_file)
                self.uuids.append(npy_file.replace('.npy', ''))
            except Exception as e:
                logger.warning(f"Failed to load {npy_file}: {e}")
                continue

        if not self.valid_files:
            logger.error(f"No valid .npy files found in {triplane_dir}")
            raise ValueError(f"No valid .npy files found in {triplane_dir}")

        logger.info(f"Initialized dataset with {len(self.valid_files)} valid samples")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        npy_file = self.valid_files[idx]
        file_path = os.path.join(self.triplane_dir, npy_file)

        try:
            data = np.load(file_path, allow_pickle=True).item()
            plane_xy = torch.from_numpy(data['plane_xy']).float().to(self.device)
            plane_yz = torch.from_numpy(data['plane_yz']).float().to(self.device)
            plane_xz = torch.from_numpy(data['plane_xz']).float().to(self.device)
            caption = str(data['caption'])
            return (plane_xy, plane_yz, plane_xz), caption
        except Exception as e:
            logger.error(f"Failed to load {npy_file}: {e}")
            raise e

class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model, height, width):
        super().__init__()
        if d_model % 4 != 0:
            raise ValueError(f"Cannot use sin/cos positional encoding with odd dimension (got dim={d_model})")
        pe = torch.zeros(d_model, height, width)
        d_model_h = d_model // 2
        d_model_w = d_model // 2
        div_term = torch.exp(torch.arange(0., d_model_h, 2) * -(torch.log(torch.tensor(10000.0)) / d_model_h))
        pos_w = torch.arange(0., width).unsqueeze(1)
        pos_h = torch.arange(0., height).unsqueeze(1)
        pe[0:d_model_h:2, :, :] = torch.sin(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
        pe[1:d_model_h:2, :, :] = torch.cos(pos_h * div_term).transpose(0, 1).unsqueeze(2).repeat(1, 1, width)
        pe[d_model_h::2, :, :] = torch.sin(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
        pe[d_model_h+1::2, :, :] = torch.cos(pos_w * div_term).transpose(0, 1).unsqueeze(1).repeat(1, height, 1)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :, :x.size(2), :x.size(3)]
        return x

class TriplaneDecoder(nn.Module):
    def __init__(self, encoder_dim=384, decoder_dim=512, decoder_layers=4, decoder_heads=8,
                 output_channels=3, output_resolution=128, input_patch_grid_res=16):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.decoder_dim = decoder_dim
        self.output_resolution = output_resolution
        self.input_patch_grid_res = input_patch_grid_res
        self.num_patches = input_patch_grid_res * input_patch_grid_res

        self.input_proj = nn.Linear(encoder_dim, decoder_dim * self.num_patches)
        self.pos_encoder = PositionalEncoding2D(decoder_dim, input_patch_grid_res, input_patch_grid_res)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=decoder_dim, nhead=decoder_heads, dim_feedforward=decoder_dim * 4,
            dropout=0.1, activation=F.gelu, batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_layers)

        num_upsample_stages = int(torch.log2(torch.tensor(output_resolution // input_patch_grid_res)))
        if input_patch_grid_res * (2**num_upsample_stages) != output_resolution:
            raise ValueError("Output resolution must be a power-of-2 multiple of input_patch_grid_res")

        upsample_layers = []
        current_dim = decoder_dim
        for i in range(num_upsample_stages):
            out_dim = max(decoder_dim // (2**(i+1)), output_channels * 2)
            upsample_layers.append(nn.Sequential(
                nn.ConvTranspose2d(current_dim, out_dim, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_dim), nn.GELU(),
                nn.Conv2d(out_dim, out_dim, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_dim), nn.GELU()
            ))
            current_dim = out_dim
        self.upsample_neck = nn.Sequential(*upsample_layers)

        self.output_proj = nn.Conv2d(current_dim, output_channels, kernel_size=1, stride=1, padding=0)
        self.output_activation = nn.Sigmoid()

    def forward(self, text_embedding):
        batch_size = text_embedding.shape[0]
        seq = self.input_proj(text_embedding).view(batch_size, self.num_patches, self.decoder_dim)
        spatial_input = seq.permute(0, 2, 1).reshape(
            batch_size, self.decoder_dim, self.input_patch_grid_res, self.input_patch_grid_res)
        spatial_input_with_pos = self.pos_encoder(spatial_input)
        seq_with_pos = spatial_input_with_pos.flatten(2).permute(0, 2, 1)
        refined_seq = self.transformer_decoder(tgt=seq_with_pos, memory=seq_with_pos)
        spatial_features = refined_seq.permute(0, 2, 1).reshape(
            batch_size, self.decoder_dim, self.input_patch_grid_res, self.input_patch_grid_res)
        upsampled_features = self.upsample_neck(spatial_features)
        output_logits = self.output_proj(upsampled_features)
        output_triplane = self.output_activation(output_logits)

        plane_xy = output_triplane[:, 0:1, :, :]
        plane_yz = output_triplane[:, 1:2, :, :]
        plane_xz = output_triplane[:, 2:3, :, :]

        return plane_xy, plane_yz, plane_xz

class TextToTriplaneModel(nn.Module):
    def __init__(self, text_encoder_name='all-MiniLM-L6-v2', decoder_dim=512, decoder_layers=4,
                 decoder_heads=8, output_resolution=128):
        super().__init__()
        self.text_encoder = SentenceTransformer(text_encoder_name)
        self.text_encoder.eval()
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        self.decoder = TriplaneDecoder(
            encoder_dim=384,
            decoder_dim=decoder_dim,
            decoder_layers=decoder_layers,
            decoder_heads=decoder_heads,
            output_channels=3,
            output_resolution=output_resolution,
            input_patch_grid_res=16
        )

    def forward(self, captions, device):
        with torch.no_grad():
            text_embeddings = self.text_encoder.encode(captions, convert_to_tensor=True, device=device)
        plane_xy, plane_yz, plane_xz = self.decoder(text_embeddings)
        return plane_xy, plane_yz, plane_xz

def plot_sample(triplane, caption, uuid, title="Triplane"):
    try:
        if any(ord(char) > 127 for char in caption):
            try:
                plt.rcParams['font.family'] = 'Noto Sans CJK JP'
            except:
                logger.warning("Noto Sans CJK JP font not found, using default.")
                plt.rcParams['font.family'] = 'DejaVu Sans'
        else:
            plt.rcParams['font.family'] = 'DejaVu Sans'

        logger.info(f"Generating plot: {uuid}_{title.replace(' ', '_').lower()}.png")

        plane_xy, plane_yz, plane_xz = triplane
        xy_plane = plane_xy.cpu().numpy()[0]
        yz_plane = plane_yz.cpu().numpy()[0]
        xz_plane = plane_xz.cpu().numpy()[0]
        xy_plane = np.clip(xy_plane, 0, 1)
        yz_plane = np.clip(yz_plane, 0, 1)
        xz_plane = np.clip(xz_plane, 0, 1)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(xy_plane, cmap='gray')
        axes[0].set_title(f"XY Plane\nUUID: {uuid}")
        axes[0].axis('off')
        axes[1].imshow(yz_plane, cmap='gray')
        axes[1].set_title("YZ Plane")
        axes[1].axis('off')
        axes[2].imshow(xz_plane, cmap='gray')
        axes[2].set_title("XZ Plane")
        axes[2].axis('off')
        fig.suptitle(f"{title}\nCaption: {caption}", fontsize=12, y=0.05)
        plt.tight_layout()
        plt.savefig(f"{uuid}_{title.replace(' ', '_').lower()}.png")
        plt.show()
        plt.close()

        plt.rcParams['font.family'] = 'DejaVu Sans'
    except Exception as e:
        logger.error(f"Error plotting sample: {e}")

def plot_triplane_grid(pred_triplane, gt_triplane, caption, uuid, title="Triplane Comparison"):
    try:
        if any(ord(char) > 127 for char in caption):
            try:
                plt.rcParams['font.family'] = 'Noto Sans CJK JP'
            except:
                logger.warning("Noto Sans CJK JP font not found, using default.")
                plt.rcParams['font.family'] = 'DejaVu Sans'
        else:
            plt.rcParams['font.family'] = 'DejaVu Sans'

        logger.info(f"Generating grid plot: {uuid}_{title.replace(' ', '_').lower()}.png with caption: {caption}")

        pred_xy, pred_yz, pred_xz = pred_triplane
        gt_xy, gt_yz, gt_xz = gt_triplane

        pred_xy = np.clip(pred_xy.cpu().numpy()[0, 0], 0, 1)
        pred_yz = np.clip(pred_yz.cpu().numpy()[0, 0], 0, 1)
        pred_xz = np.clip(pred_xz.cpu().numpy()[0, 0], 0, 1)
        gt_xy = np.clip(gt_xy.cpu().numpy()[0], 0, 1)
        gt_yz = np.clip(gt_yz.cpu().numpy()[0], 0, 1)
        gt_xz = np.clip(gt_xz.cpu().numpy()[0], 0, 1)

        fig, axes = plt.subplots(2, 3, figsize=(15, 10))

        axes[0, 0].imshow(pred_xy, cmap='gray')
        axes[0, 0].set_title("Predicted XY Plane")
        axes[0, 0].axis('off')
        axes[0, 1].imshow(pred_yz, cmap='gray')
        axes[0, 1].set_title("Predicted YZ Plane")
        axes[0, 1].axis('off')
        axes[0, 2].imshow(pred_xz, cmap='gray')
        axes[0, 2].set_title("Predicted XZ Plane")
        axes[0, 2].axis('off')

        axes[1, 0].imshow(gt_xy, cmap='gray')
        axes[1, 0].set_title("Ground Truth XY Plane")
        axes[1, 0].axis('off')
        axes[1, 1].imshow(gt_yz, cmap='gray')
        axes[1, 1].set_title("Ground Truth YZ Plane")
        axes[1, 1].axis('off')
        axes[1, 2].imshow(gt_xz, cmap='gray')
        axes[1, 2].set_title("Ground Truth XZ Plane")
        axes[1, 2].axis('off')

        fig.suptitle(f"{title}\nCaption: {caption}\nUUID: {uuid}", fontsize=12, y=0.02)
        plt.tight_layout()
        plt.savefig(f"{uuid}_{title.replace(' ', '_').lower()}.png")
        plt.show()
        plt.close()

        plt.rcParams['font.family'] = 'DejaVu Sans'
    except Exception as e:
        logger.error(f"Error plotting triplane grid: {e}")
        raise e

def train_model(model, train_loader, val_loader, npy_file_path, num_epochs=10, lr=1e-3, device='cuda', checkpoint_dir='.'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = nn.L1Loss()

    best_val_loss = float('inf')
    best_model_path = os.path.join(checkpoint_dir, "best_text_to_triplane_model.pth")

    os.makedirs(checkpoint_dir, exist_ok=True)

    logger.info(f"Loading .npy file: {npy_file_path}")
    try:
        data = np.load(not all(key in data for key in ['plane_xy', 'plane_yz', 'plane_xz', 'caption']))
        if data['plane_xy'].shape != (1, 128, 128) or \
           data['plane_yz'].shape != (1, 128, 128) or \
           data['plane_xz'].shape != (1, 128, 128):
            raise ValueError(f"Invalid .npy file {npy_file_path}: incorrect triplane shapes")
        npy_caption = str(data['caption'])
        npy_gt_xy = torch.from_numpy(data['plane_xy']).float()
        npy_gt_yz = torch.from_numpy(data['plane_yz']).float()
        npy_gt_xz = torch.from_numpy(data['plane_xz']).float()
        logger.info(f"Successfully loaded .npy file with caption: {npy_caption}")
    except Exception as e:
        logger.error(f"Failed to load {npy_file_path}: {e}")
        raise e

    prev_pred_triplane = None

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for triplane, caption in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
            triplane_xy, triplane_yz, triplane_xz = triplane
            triplane_xy = triplane_xy.to(device)
            triplane_yz = triplane_yz.to(device)
            triplane_xz = triplane_xz.to(device)
            optimizer.zero_grad()
            output_xy, output_yz, output_xz = model(caption, device=device)
            loss_xy = criterion(output_xy, triplane_xy)
            loss_yz = criterion(output_yz, triplane_yz)
            loss_xz = criterion(output_xz, triplane_xz)
            loss = (loss_xy + loss_yz + loss_xz) / 3
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            torch.cuda.empty_cache() if device == 'cuda' else None
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for triplane, caption in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
                triplane_xy, triplane_yz, triplane_xz = triplane
                triplane_xy = triplane_xy.to(device)
                triplane_yz = triplane_yz.to(device)
                triplane_xz = triplane_xz.to(device)
                output_xy, output_yz, output_xz = model(caption, device=device)
                loss_xy = criterion(output_xy, triplane_xy)
                loss_yz = criterion(output_yz, triplane_yz)
                loss_xz = criterion(output_xz, triplane_xz)
                loss = (loss_xy + loss_yz + loss_xz) / 3
                val_loss += loss.item()
        val_loss /= len(val_loader)

        scheduler.step()

        logger.info(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            logger.info(f"Saved best model with Val Loss: {best_val_loss:.4f}")

        if (epoch + 1) % 10 == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
            torch.save(model.state_dict(), checkpoint_path)
            logger.info(f"Saved checkpoint at epoch {epoch+1} to {checkpoint_path}")

            logger.info(f"Generating triplanes for .npy file caption at epoch {epoch+1}")
            model.eval()
            with torch.no_grad():
                output_xy, output_yz, output_xz = model([npy_caption], device=device)
                pred_triplane = (output_xy, output_yz, output_xz)
                gt_triplane = (npy_gt_xy, npy_gt_yz, npy_gt_xz)

                if prev_pred_triplane is not None:
                    diff_xy = torch.mean(torch.abs(output_xy - prev_pred_triplane[0])).item()
                    diff_yz = torch.mean(torch.abs(output_yz - prev_pred_triplane[1])).item()
                    diff_xz = torch.mean(torch.abs(output_xz - prev_pred_triplane[2])).item()
                    logger.info(f"Prediction L1 diff between epoch {epoch+1} and {epoch-9}: "
                                f"XY={diff_xy:.6f}, YZ={diff_yz:.6f}, XZ={diff_xz:.6f}")

                prev_pred_triplane = (output_xy.clone(), output_yz.clone(), output_xz.clone())

                plot_triplane_grid(
                    pred_triplane=pred_triplane,
                    gt_triplane=gt_triplane,
                    caption=npy_caption,
                    uuid=f"inference_{os.path.basename(npy_file_path).replace('.npy', '')}_epoch_{epoch+1}",
                    title=f"Triplane Comparison Epoch {epoch+1}"
                )

    return model, best_model_path

def load_model_checkpoint(checkpoint_path, device='cuda', train_mode=False):
    logger = logging.getLogger(__name__)
    try:
        model = TextToTriplaneModel(
            text_encoder_name='all-MiniLM-L6-v2',
            decoder_dim=512,
            decoder_layers=4,
            decoder_heads=8,
            output_resolution=128
        )
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint)
        logger.info(f"Successfully loaded checkpoint from {checkpoint_path}")
        model = model.to(device)
        if train_mode:
            model.train()
            logger.info("Model set to training mode")
        else:
            model.eval()
            logger.info("Model set to evaluation mode")
        return model
    except FileNotFoundError:
        logger.error(f"Checkpoint file not found: {checkpoint_path}")
        raise
    except Exception as e:
        logger.error(f"Failed to load checkpoint from {checkpoint_path}: {e}")
        raise

def infer_triplane(model, caption, device='cuda'):
    model.eval()
    uuid = f"inference_{hash(caption) % 1000000}"
    with torch.no_grad():
        output_xy, output_yz, output_xz = model([caption], device=device)
        triplane = (output_xy[0], output_yz[0], output_xz[0])
        plot_sample(triplane, caption, uuid, title="Inferred Triplane")
        logger.info(f"Plotted inferred triplane for caption: {caption}")
    return triplane

# **Configurations**

In [9]:
# Configuration
CAPTION_DATA_PATH = "/content/drive/MyDrive/Project/df_english.npy"
TRIPLANE_DIR = "/content/drive/MyDrive/Project/triplanes"
CHECKPOINT_DIR = "/content/drive/MyDrive/Project/checkpoints"  # New directory for checkpoints
NPY_FILE_PATH = "/content/drive/MyDrive/Project/triplanes/fecabaeaf1794756b45333b97d6e2374.npy"
START_INDEX = 648
END_INDEX = 700
NUM_MODELS = 5000
TRAIN_SPLIT = 0.75
VAL_SPLIT = 0.20
TEST_SPLIT = 0.05
BATCH_SIZE = 1
NUM_EPOCHS = 100
LEARNING_RATE = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'



# **Create the dataset**

In [10]:
# # Generate triplanes
# generate_and_save_triplanes(
#     caption_data_path=CAPTION_DATA_PATH,
#     output_dir=TRIPLANE_DIR,
#     start_index=START_INDEX,
#     end_index=END_INDEX,
#     triplane_resolution=128,
#     voxel_resolution=128,
#     device=DEVICE
# )

In [12]:
# def count_files(directory):
#     return len([f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])

# num_files = count_files(TRIPLANE_DIR)
# print(f"Number of files in {TRIPLANE_DIR}: {num_files}")

# **Load the Dataset**

In [24]:
# Create dataset
dataset = TriplaneTextFromFilesDataset(
    triplane_dir=TRIPLANE_DIR,
    num_models=NUM_MODELS,
    device=DEVICE
)

# Split dataset
train_size = int(TRAIN_SPLIT * len(dataset))
val_size = int(VAL_SPLIT * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Validating .npy files: 100%|██████████| 5000/5000 [01:29<00:00, 55.96it/s] 


In [14]:
# best_model_path = f"{CHECKPOINT_DIR}/checkpoint_epoch_40.pth"
# model = load_model_checkpoint(best_model_path, device=DEVICE, train_mode=False)

# **Load the model and train**

In [26]:
model = TextToTriplaneModel(
    text_encoder_name='all-MiniLM-L6-v2',
    decoder_dim=512,
    decoder_layers=4,
    decoder_heads=8,
    output_resolution=128
)

trained_model, best_model_path = train_model(
    model, train_loader, val_loader,
    npy_file_path=NPY_FILE_PATH,
    num_epochs=NUM_EPOCHS, lr=LEARNING_RATE, device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR
)

# **Chamfer Distance calculation**

In [ ]:


class TextToTriplaneModel(nn.Module):
    """
    A model that converts text captions to triplanes (XY, YZ, XZ planes).
    """
    def __init__(self, text_encoder_name='all-MiniLM-L6-v2', decoder_dim=512,
                 decoder_layers=4, decoder_heads=8, output_resolution=128):
        super().__init__()
        # Initialize text encoder
        self.text_encoder = SentenceTransformer(text_encoder_name).to(DEVICE)
        # Simple decoder
        self.decoder = nn.Sequential(
            nn.Linear(384, decoder_dim),  # Assuming text encoder outputs 384-dim embeddings
            nn.ReLU(),
            nn.Linear(decoder_dim, 3 * output_resolution * output_resolution)  # Output 3 planes
        ).to(DEVICE)
        self.output_resolution = output_resolution

        # Move model to device
        self.to(DEVICE)

    def forward(self, captions, device=DEVICE):
        """
        Forward pass: Convert captions to triplanes.
        """
        try:
            # Encode captions to embeddings
            embeddings = self.text_encoder.encode(captions, convert_to_tensor=True)
            embeddings = embeddings.to(device)
            logger.debug(f"Text embeddings device: {embeddings.device}")

            # Decode to triplanes
            output = self.decoder(embeddings)
            # Reshape to (batch_size, 1, output_resolution, output_resolution) for each plane
            batch_size = embeddings.size(0)
            planes = output.view(batch_size, 3, 1, self.output_resolution, self.output_resolution)
            pred_xy, pred_yz, pred_xz = planes[:, 0], planes[:, 1], planes[:, 2]

            return pred_xy, pred_yz, pred_xz
        except Exception as e:
            logger.error(f"Error in TextToTriplaneModel forward: {e}")
            raise

def triplane_to_point_cloud(triplane, threshold=0.5, resolution=128, device=DEVICE):
    """
    Convert a triplane to a 3D point cloud.
    """
    plane_xy, plane_yz, plane_xz = triplane
    batch_size = plane_xy.shape[0]
    point_clouds = []

    for i in range(batch_size):
        xy = plane_xy[i].squeeze(0)
        yz = plane_yz[i].squeeze(0)
        xz = plane_xz[i].squeeze(0)

        # Log triplane statistics
        logger.debug(f"Triplane {i} - XY: min={xy.min().item():.4f}, max={xy.max().item():.4f}, mean={xy.mean().item():.4f}")
        logger.debug(f"Triplane {i} - YZ: min={yz.min().item():.4f}, max={yz.max().item():.4f}, mean={yz.mean().item():.4f}")
        logger.debug(f"Triplane {i} - XZ: min={xz.min().item():.4f}, max={xz.max().item():.4f}, mean={xz.mean().item():.4f}")

        # Dynamic threshold
        max_val = max(xy.max().item(), yz.max().item(), xz.max().item())
        threshold_i = threshold
        if max_val < threshold:
            threshold_i = max_val * 0.5 if max_val > 0 else 0.1
            logger.info(f"Triplane {i} max value {max_val:.4f} < threshold {threshold}. Using dynamic threshold: {threshold_i:.4f}")

        xy = (xy > threshold_i).float()
        yz = (yz > threshold_i).float()
        xz = (xz > threshold_i).float()

        points = []

        if xy.sum() > 0:
            y, x = torch.where(xy > 0)
            z = torch.zeros_like(x, dtype=torch.float32, device=device)
            x = (x.float() / (resolution - 1)) * 2 - 1
            y = (y.float() / (resolution - 1)) * 2 - 1
            xy_points = torch.stack([x, y, z], dim=-1)
            points.append(xy_points)

        if yz.sum() > 0:
            z, y = torch.where(yz > 0)
            x = torch.zeros_like(y, dtype=torch.float32, device=device)
            y = (y.float() / (resolution - 1)) * 2 - 1
            z = (z.float() / (resolution - 1)) * 2 - 1
            yz_points = torch.stack([x, y, z], dim=-1)
            points.append(yz_points)

        if xz.sum() > 0:
            z, x = torch.where(xz > 0)
            y = torch.zeros_like(x, dtype=torch.float32, device=device)
            x = (x.float() / (resolution - 1)) * 2 - 1
            z = (z.float() / (resolution - 1)) * 2 - 1
            xz_points = torch.stack([x, y, z], dim=-1)
            points.append(xz_points)

        if not points:
            logger.warning(f"No points found in triplane {i} after thresholding with threshold {threshold_i}")
            point_clouds.append(torch.empty((0, 3), device=device))
        else:
            point_clouds.append(torch.cat(points, dim=0))

    return point_clouds

def chamfer_distance(pc1, pc2, device=DEVICE):
    """
    Compute the Chamfer Distance between two point clouds.
    """
    if pc1.shape[0] == 0 or pc2.shape[0] == 0:
        logger.warning("One or both point clouds are empty")
        return float('inf')

    pc1 = pc1.unsqueeze(1)
    pc2 = pc2.unsqueeze(0)
    dist = torch.sum((pc1 - pc2) ** 2, dim=-1)

    dist1 = dist.min(dim=1)[0]
    dist2 = dist.min(dim=0)[0]

    loss1 = dist1.mean()
    loss2 = dist2.mean()

    return (loss1 + loss2).item()

def plot_chamfer_distances(distances_data, output_path="chamfer_distances.png"):
    """
    Plot Chamfer Distances for each data sample, suitable for Colab.
    """
    if not distances_data:
        logger.error("No valid Chamfer Distances to plot")
        return

    indices = [data[0] for data in distances_data]
    distances = [data[1] for data in distances_data]

    plt.figure(figsize=(12, 6))
    plt.scatter(indices, distances, color='red', alpha=0.5, label='Chamfer Distance')
    plt.xlabel('Sample Index')
    plt.ylabel('Chamfer Distance')
    plt.title('Chamfer Distance per Validation Sample')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()  # Display plot in Colab
    logger.info(f"Saved Chamfer Distance plot to {output_path}")

def evaluate_chamfer_distance(model, val_loader, device=DEVICE, threshold=0.5,
                             resolution=128, max_samples=None, plot_path="chamfer_distances.png"):
    """
    Evaluate Chamfer Distance on the validation dataset.
    """
    model.eval()
    chamfer_distances = []
    distances_data = []
    sample_count = 0

    # Ensure model is on the correct device
    model.to(device)
    logger.info("Model moved to device: %s", device)

    with torch.no_grad():
        for triplane, captions in tqdm(val_loader, desc="Evaluating Validation Pairs"):
            try:
                # Move ground truth triplanes to device
                gt_triplane = (
                    triplane[0].to(device),
                    triplane[1].to(device),
                    triplane[2].to(device)
                )

                # Generate predicted triplanes
                pred_xy, pred_yz, pred_xz = model(captions, device=device)
                pred_triplane = (pred_xy, pred_yz, pred_xz)

                # Log predicted triplane statistics
                logger.info(f"Pred XY: min={pred_xy.min().item():.4f}, max={pred_xy.max().item():.4f}, mean={pred_xy.mean().item():.4f}")
                logger.info(f"Pred YZ: min={pred_yz.min().item():.4f}, max={pred_yz.max().item():.4f}, mean={pred_yz.mean().item():.4f}")
                logger.info(f"Pred XZ: min={pred_xz.min().item():.4f}, max={pred_xz.max().item():.4f}, mean={pred_xz.mean().item():.4f}")

                # Convert triplanes to point clouds
                gt_points_list = triplane_to_point_cloud(
                    gt_triplane, threshold=threshold, resolution=resolution, device=device
                )
                pred_points_list = triplane_to_point_cloud(
                    pred_triplane, threshold=threshold, resolution=resolution, device=device
                )

                # Compute Chamfer Distances
                for i, (gt_points, pred_points, caption) in enumerate(zip(gt_points_list, pred_points_list, captions)):
                    logger.info(f"Sample {sample_count} - GT Points: {gt_points.shape[0]}, Pred Points: {pred_points.shape[0]}")
                    cd = chamfer_distance(gt_points, pred_points, device=device)
                    if np.isfinite(cd):
                        chamfer_distances.append(cd)
                        distances_data.append((sample_count, cd, caption))
                        logger.info(f"Caption: {caption[:50]}..., Chamfer Distance: {cd:.6f}")
                    else:
                        logger.warning(f"Invalid Chamfer Distance for caption: {caption[:50]}...")

                    sample_count += 1
                    if max_samples and sample_count >= max_samples:
                        break

                # Clean up
                del gt_points_list, pred_points_list, pred_xy, pred_yz, pred_xz
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

                if max_samples and sample_count >= max_samples:
                    break

            except Exception as e:
                logger.error(f"Error processing batch with captions {[c[:50] for c in captions]}...: {e}")
                continue

    # Summarize results
    if chamfer_distances:
        max_cd = max(chamfer_distances)
        min_cd = min(chamfer_distances)
        avg_cd = np.mean(chamfer_distances)
        logger.info(f"Chamfer Distance Statistics: Max={max_cd:.6f}, Min={min_cd:.6f}, Avg={avg_cd:.6f}")
    else:
        logger.error("No valid Chamfer Distances computed")
        max_cd, min_cd, avg_cd = None, None, None

    return {
        'max_chamfer': max_cd,
        'min_chamfer': min_cd,
        'avg_chamfer': avg_cd,
        'distances': chamfer_distances,
        'distances_data': distances_data
    }

# Sample Dataset for Testing
class TriplaneDataset(Dataset):
    """
    A sample dataset returning synthetic triplanes and captions for testing.
    """
    def __init__(self, num_samples=10, resolution=128):
        self.num_samples = num_samples
        self.resolution = resolution
        self.captions = [f"Sample object {i}" for i in range(num_samples)]
        # Synthetic triplanes with controlled values to ensure non-empty point clouds
        self.triplanes = [
            torch.sigmoid(torch.randn(3, 1, resolution, resolution)) * 2  # Values in [0, 2]
            for _ in range(num_samples)
        ]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        triplane = self.triplanes[idx]
        caption = self.captions[idx]
        return triplane, caption


# Verify model parameters
logger.info("Checking model parameter devices:")
for name, param in model.named_parameters():
    logger.info(f"Parameter {name} is on device: {param.device}")

# Evaluate
results = evaluate_chamfer_distance(
    model=model,
    val_loader=val_loader,
    device=DEVICE,
    threshold=0.5,
    resolution=128,
    max_samples=1000,
    plot_path="chamfer_distances.png"
)

# Print and plot results
print(f"Chamfer Distance Results:")
if results['max_chamfer'] is not None:
    print(f"Maximum: {results['max_chamfer']:.6f}")
    print(f"Minimum: {results['min_chamfer']:.6f}")
    print(f"Average: {results['avg_chamfer']:.6f}")
    # Plot the results
    plot_chamfer_distances(results['distances_data'], output_path="chamfer_distances.png")
else:
    print("No valid Chamfer Distances computed. Check device mismatch errors.")